This script extracts the mean spectroscopic data from the reconstructed photoacoustic images.

In [ ]:
import pandas as pd
import numpy as np

import patato as pat

from paiskintonetools.correction_factor import get_correction_factor_interpolator  # type: ignore
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
df_scans = pd.read_parquet("scan_table.parquet")

In [ ]:
df_scans[["SkinID", "ScanNumber"]]

In [ ]:
# Load the estimates of melanosome volume fraction derived from the photoacoustic model
pa_mvf_file = Path("tables/pa_derived_mvf_estimate.parquet")
if not pa_mvf_file.exists():
    print(
        "To generate results with pa-derived mvf predictions, first run this script through, then run pai_based_correction.ipynb, then rerun this."
    )
else:
    df_pa_mvf = pd.read_parquet(pa_mvf_file)
    df_scans = df_scans.merge(df_pa_mvf, on=["SkinID", "ScanNumber"], validate="1:1")

In [ ]:
cal_curve_file = "../Fluence Correction/tables/cali_curve.csv"
correction_factor_spline = get_correction_factor_interpolator(cal_curve_file)

In [ ]:
def unmix(rec):
    wl = rec.wavelengths[rec.wavelengths < 900]
    um = pat.SpectralUnmixer(chromophores=["Hb", "HbO2"], wavelengths=wl)
    u, _, _ = um.run(rec, None)

    thber = pat.THbCalculator()
    so2er = pat.SO2Calculator()

    thb, _, _ = thber.run(u, None)
    so2, _, _ = so2er.run(u, None)

    return thb, so2


def get_measurements(file, row):
    # Note: when first running this script on new data you would have to run without pa-based to get the initial spectral means, then run pai_based_correction.ipynb to generate the appropriate lookup table.
    if "PADerivedMVFEstimate" not in row:
        correction_methods = ["none", "ita-based"]
    else:
        correction_methods = ["none", "ita-based", "pa-based"]

    pa = pat.PAData.from_hdf5(file)
    pa.set_default_recon(("Model Based", "0"))  # type: ignore
    results = []
    ita = row["ITA"]
    for n, roi in pa.get_rois().items():
        row_output = dict(row)
        row_output["ROI Name"] = " ".join(n[0].split("_"))
        row_output["ROI Number"] = n[1]
        for correct in correction_methods:
            # Apply correction factor or not
            rec = pa.get_scan_reconstructions()
            rec = rec.copy()
            rec.raw_data = np.copy(rec.raw_data)  # type: ignore
            if correct == "ita-based":
                mvf = (19.028 - 0.3692 * ita + 0.001685 * ita**2) / 100
                rec.raw_data *= np.exp(  # type: ignore
                    correction_factor_spline((rec.wavelengths, mvf))[  # type: ignore
                        None, :, None, None, None
                    ]
                )
            elif correct == "pa-based":
                # Correct based on pa-derived mvf predictions.
                mvf = row["PADerivedMVFEstimate"]
                rec.raw_data *= np.exp(  # type: ignore
                    correction_factor_spline((rec.wavelengths, mvf))[  # type: ignore
                        None, :, None, None, None
                    ]
                )

            thb, so2 = unmix(rec)
            roi_x = roi.get_polygon().centroid.x
            roi_y = roi.get_polygon().centroid.y

            row_output["ROI x"] = roi_x
            row_output["ROI y"] = roi_y

            row_output["ROI area"] = roi.get_polygon().area
            for measurement, positive in [
                ("spectrum", False),
                ("spectrum", True),
                ("thb", False),
                ("thb", True),
                ("so2", None),
            ]:
                if measurement == "spectrum":
                    calc = rec
                elif measurement == "thb":
                    calc = thb
                elif measurement == "so2":
                    calc = so2
                if measurement == "so2":
                    r = calc.raw_data  # type: ignore
                    r[thb.raw_data < 0] = np.nan  # type: ignore
                    r[r > 1.5] = np.nan  # type: ignore
                    r[r < 0] = np.nan  # type: ignore
                elif positive:
                    r = calc.raw_data  # type: ignore
                    r[r < 0] = np.nan  # type: ignore

                mask, _ = roi.to_mask_slice(calc)  # type: ignore
                if measurement == "spectrum" and positive:
                    dx = float(calc.da.coords["x"][1] - calc.da.coords["x"][0])  #  type:ignore
                    row_output["ROI area"] = np.sum(mask & ~np.isnan(r)) * dx**2  # type: ignore

                s = np.squeeze(calc.raw_data.T[mask.T].T)  # type: ignore

                for agg in [np.nanmean, np.nanmedian, np.nanstd]:
                    n = agg.__name__[3:]
                    f = (
                        (
                            "corrected_"
                            if correct == "ita-based"
                            else "pacorrection"
                            if correct == "pa-based"
                            else ""
                        )
                        + measurement
                        + ("_positive" if positive else "")
                        + "_"
                        + n
                    )
                    row_output[f] = agg(s, axis=-1)
        results.append(row_output)
    return pd.DataFrame(results)

In [ ]:
import warnings

# N.b. catching warnings because sometimes no pixels contain positive values
with warnings.catch_warnings(action="ignore"):
    scans = []
    for _, row in tqdm(df_scans.iterrows(), total=df_scans.shape[0]):
        scans.append(get_measurements(row["File"], row))

In [ ]:
df_result = pd.concat(scans)

In [ ]:
df_result["ROI Name"] = df_result["ROI Name"].str.strip()

In [ ]:
df_result.to_parquet("tables/pa_values_extracted.parquet")